# Experiment Figures

Manuscript figures from completed CelebA and LFWA runs. LFWA quantitative plots use
`results/lfwa_full_sweep`: ten positive anchor-noise levels and ten positive private-noise
levels, with four realizations within each of five deployment seeds. Figure 3 keeps
separate LFWA and CelebA panels; gallery annotations are matched seed-0, realization-0
linkage scores, rather than five-seed means. No utility model fitting.

For an LFWA-only update from saved results, mount Drive (cell 1), then run the final
**Refresh LFWA figures only** cell. It validates run coverage, completion, saved
statistics, gallery provenance, and GPM convergence flags before rendering.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
"""Standalone CelebA protocol, MLP, and frozen reconstruction-auditor component.

Records are row vectors. Protocol arithmetic is float64; utility and face models
use float32. Rates returned by this module are fractions, not percentages.
"""

import hashlib
import json
import os
import random
import time
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
from torch import nn
from torch.nn import functional as torch_functional


def _model_seed(seed, *parts):
    payload = json.dumps([int(seed), *parts], separators=(",", ":"), ensure_ascii=True)
    return int.from_bytes(hashlib.sha256(payload.encode()).digest()[:4], "little")


def _model_numpy(value, dtype=np.float64):
    if torch.is_tensor(value):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=dtype)


def _model_sync(device):
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)


def _model_haar(rng, h):
    q, r = np.linalg.qr(rng.standard_normal((h, h)))
    signs = np.where(np.diag(r) >= 0.0, 1.0, -1.0)
    return q * signs[None, :]


def generate_bank(seed, pmax=50, h=400, r=1600):
    """Stable participant-indexed draws; changing pmax cannot change a prefix.

    A is centered and satisfies A.T @ A = I. O_i are independent Haar draws
    on O(h); translations have independent N(0,1) coordinates.
    """
    if not (1 <= h < r and pmax >= 1):
        raise ValueError("Require 1 <= h < r and pmax >= 1.")
    rng = np.random.default_rng(_model_seed(seed, "anchor"))
    a = rng.standard_normal((r, h))
    a -= a.mean(axis=0)
    a, triangular = np.linalg.qr(a, mode="reduced")
    a *= np.where(np.diag(triangular) >= 0, 1.0, -1.0)[None, :]
    rotations, translations = [], []
    for participant in range(pmax):
        rotations.append(_model_haar(np.random.default_rng(
            _model_seed(seed, "rotation", participant)), h))
        translations.append(np.random.default_rng(
            _model_seed(seed, "translation", participant)).standard_normal(h))
    return {"A": a, "O": np.stack(rotations), "psi": np.stack(translations),
            "seed": int(seed), "h": int(h), "r": int(r), "pmax": int(pmax)}


def noisy_anchor_bank(bank, seed, level_index, realization, v):
    """One alternative release. Noise is iid over entries and participants.

    A level/realization's participant prefix is independent of requested p.
    This does not center W before release: its mean perturbs the translation.
    """
    if not np.isfinite(v) or v < 0:
        raise ValueError("Anchor scale must be finite and nonnegative.")
    if "clean_anchors" not in bank:
        bank["clean_anchors"] = [bank["A"] @ o + psi
                                 for o, psi in zip(bank["O"], bank["psi"])]
    if v == 0:
        return [b.copy() for b in bank["clean_anchors"]]
    return [b + float(v) * np.random.default_rng(_model_seed(
        seed, "anchor-noise", int(level_index), int(realization), participant
    )).standard_normal(b.shape)
        for participant, b in enumerate(bank["clean_anchors"])]


def upload_blocks(blocks, bank):
    """Create private-data uploads without perturbation, Y_i = Q_i O_i + psi_i."""
    return [_model_numpy(q) @ bank["O"][i] + bank["psi"][i]
            for i, q in enumerate(blocks)]


def _model_polar(matrix):
    u, _, vh = torch.linalg.svd(matrix, full_matrices=False)
    return u @ vh


def align_anchors(B, device="cuda", tol=1e-7, max_iters=500, reference=0):
    """Ling spectral initialization and unregularized generalized power method.

    No (p*h)-square Gram is allocated. If r < p*h, spectral initialization
    diagonalizes D D.T and recovers right singular vectors as D.T U / s.
    Iteration is exactly R_next = block_polar(D.T D R), as in the old code.
    Stopping uses the OLD absolute Frobenius step <= tol (not relative step).
    The common output gauge is normalized to the reference participant.

    Returns (maps, diagnostics), with maps['rotations_R'] of shape (p,h,h)
    and maps['means_b'] of shape (p,h). Aligned data are
    (Y_i-b_i) @ maps['rotations_R'][i] + b_reference.
    """
    dev = torch.device(device)
    if not B or max_iters < 1 or tol <= 0 or not 0 <= reference < len(B):
        raise ValueError("Invalid anchors, reference, or solver settings.")
    _model_sync(dev)
    started = time.perf_counter()
    blocks = [torch.as_tensor(b, dtype=torch.float64, device=dev) for b in B]
    r, h = blocks[0].shape
    p = len(blocks)
    if any(tuple(b.shape) != (r, h) for b in blocks) or r <= h:
        raise ValueError("Every anchor block must have common shape r x h, with r > h.")
    means = torch.stack([b.mean(dim=0) for b in blocks])
    D = torch.cat([b - b.mean(dim=0) for b in blocks], dim=1)
    del blocks
    if r < p * h:
        left_gram = D @ D.T
        eigenvalues, eigenvectors = torch.linalg.eigh(left_gram)
        eigenvalues = eigenvalues.flip(0)
        top = eigenvalues[:h]
        if float(top[-1].item()) <= torch.finfo(torch.float64).eps * float(top[0].item()):
            raise RuntimeError("Spectral initialization has a numerically deficient top-h subspace.")
        right = (D.T @ eigenvectors[:, -h:].flip(1)) / top.sqrt().unsqueeze(0)
        gap = float((eigenvalues[h - 1] - eigenvalues[h]).item())
        scale = float(top[-1].item())
        del eigenvectors, eigenvalues, left_gram
    else:
        _, singular, vh = torch.linalg.svd(D, full_matrices=False)
        right = vh[:h].T
        scale = float(singular[h - 1].square().item())
        next_value = float(singular[h].square().item()) if len(singular) > h else 0.0
        gap = scale - next_value
        del vh, singular
    stacked = _model_polar(right.reshape(p, h, h)).reshape(p * h, h)
    del right
    _model_sync(dev)
    initialization_seconds = time.perf_counter() - started
    objective_history = [float((D @ stacked).square().sum().item())]
    step_history = []
    converged = False
    iteration_started = time.perf_counter()
    for _ in range(max_iters):
        next_stacked = _model_polar((D.T @ (D @ stacked)).reshape(p, h, h)).reshape(p*h, h)
        step = float(torch.linalg.vector_norm(next_stacked - stacked).item())
        step_history.append(step)
        stacked = next_stacked
        objective_history.append(float((D @ stacked).square().sum().item()))
        if step <= tol:
            converged = True
            break
    _model_sync(dev)
    iteration_seconds = time.perf_counter() - iteration_started
    next_stacked = _model_polar((D.T @ (D @ stacked)).reshape(p, h, h)).reshape(p*h, h)
    fixed_absolute = float(torch.linalg.vector_norm(next_stacked - stacked).item())
    fixed_relative = fixed_absolute / max(1.0, float(torch.linalg.vector_norm(stacked).item()))
    rotations = stacked.reshape(p, h, h)
    # This exact gauge normalization is also applied to the aligned mean.
    rotations = rotations @ rotations[reference].T.clone()
    rotations[reference] = torch.eye(h, device=dev, dtype=torch.float64)
    template = sum((D[:, i*h:(i+1)*h] @ rotations[i] for i in range(p))) / p
    residual = sum(float((D[:, i*h:(i+1)*h] @ rotations[i] - template).square().sum().item())
                   for i in range(p))
    denominator = float(D.square().sum().item())
    maps = {"rotations_R": rotations.detach().cpu().numpy(),
            "means_b": means.detach().cpu().numpy(), "reference": int(reference)}
    _model_sync(dev)
    diagnostics = {
        "algorithm": "spectral-unregularized-GPM",
        "dtype": "float64", "max_iters": int(max_iters), "tolerance": float(tol),
        "stopping_rule": "absolute Frobenius norm of successive stacked iterates",
        "iterations": len(step_history), "converged": bool(converged),
        "termination_reason": "step_tolerance" if converged else "iteration_cap",
        "last_step": step_history[-1], "fixed_point_residual": fixed_relative,
        "fixed_point_residual_absolute": fixed_absolute,
        "gopp_residual": residual, "relative_gopp_residual": residual / max(denominator, 1e-300),
        "eigengap": gap, "relative_eigengap": gap / max(scale, 1e-300),
        "objective_history": objective_history, "step_history": step_history,
        "initialization_seconds": initialization_seconds, "iteration_seconds": iteration_seconds,
        "elapsed_seconds": time.perf_counter() - started,
        "reference_participant": int(reference), "participant_count": int(p),
    }
    return maps, diagnostics


def apply_maps(uploads, maps):
    means, rotations = maps["means_b"], maps["rotations_R"]
    reference = maps.get("reference", 0)
    return [(_model_numpy(y) - means[i]) @ rotations[i] + means[reference]
            for i, y in enumerate(uploads)]


def op_reconstruct(A, B, Y, device="cuda"):
    """Known-anchor OP recovery of projected records; no image decoder here."""
    dev = torch.device(device)
    a = torch.as_tensor(A, device=dev, dtype=torch.float64)
    b = torch.as_tensor(B, device=dev, dtype=torch.float64)
    y = torch.as_tensor(Y, device=dev, dtype=torch.float64)
    am, bm = a.mean(0), b.mean(0)
    rotation = _model_polar((a - am).T @ (b - bm))
    translation = bm - am @ rotation
    return ((y - translation) @ rotation.T).cpu().numpy()


def _model_set_seed(seed):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(int(seed))
    np.random.seed(int(seed) % (2**32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def _model_metrics(logits, labels):
    prediction = np.asarray(logits).reshape(-1) >= 0.0
    truth = np.asarray(labels).reshape(-1).astype(bool)
    counts = [int((truth == label).sum()) for label in (False, True)]
    if not all(counts):
        raise ValueError("Balanced accuracy requires both classes in the evaluation split.")
    recall = [float((prediction[truth == label] == label).mean()) for label in (False, True)]
    return {"balanced_accuracy": float(np.mean(recall)),
            "accuracy": float((prediction == truth).mean()),
            "recall_not_smiling": recall[0], "recall_smiling": recall[1],
            "n_records": int(len(truth))}


def fit_mlp(trainX, trainY, valX, valY, testX, testY, seed, cfg=None):
    """Fit the fixed vector MLP and select the epoch using validation BA only.

    seed must be held fixed within a deployment for paired protocol/noise runs.
    A fresh local permutation generator is reset for each call. No feature-wise
    normalization or test-dependent threshold selection is performed.
    """
    config = {"device": "cuda" if torch.cuda.is_available() else "cpu",
              "max_epochs": 50, "patience": 8, "batch_size": 256,
              "lr": 1e-3, "weight_decay": 0.0, "hidden_dims": (256, 128)}
    if cfg is not None:
        config.update(cfg if isinstance(cfg, dict) else vars(cfg))
    if int(config["max_epochs"]) < 1 or int(config["patience"]) < 1:
        raise ValueError("max_epochs and patience must be positive.")
    _model_set_seed(seed)
    dev = torch.device(config["device"])
    arrays = [_model_numpy(x, np.float32) for x in (trainX, valX, testX)]
    labels = [_model_numpy(y, np.float32).reshape(-1) for y in (trainY, valY, testY)]
    h = arrays[0].shape[1]
    if any(x.ndim != 2 or x.shape[1] != h or len(x) != len(y)
           or not np.isfinite(x).all() or not np.isin(y, [0.0, 1.0]).all()
           for x, y in zip(arrays, labels)):
        raise ValueError("MLP requires finite n x h inputs and matching binary 0/1 labels.")
    for y in labels:
        if len(np.unique(y)) != 2:
            raise ValueError("Each utility split must contain both smiling classes.")
    hidden = tuple(config["hidden_dims"])
    if len(hidden) != 2:
        raise ValueError("Expected exactly two MLP hidden layers.")
    network = nn.Sequential(nn.Linear(h, hidden[0]), nn.ReLU(),
                            nn.Linear(hidden[0], hidden[1]), nn.ReLU(),
                            nn.Linear(hidden[1], 1)).to(dev)
    optimizer = torch.optim.Adam(network.parameters(), lr=float(config["lr"]),
                                 weight_decay=float(config["weight_decay"]))
    criterion = nn.BCEWithLogitsLoss()
    xtrain = torch.from_numpy(np.ascontiguousarray(arrays[0]))
    ytrain = torch.from_numpy(labels[0].copy())
    permutation_generator = torch.Generator(device="cpu").manual_seed(int(seed))
    batch_size = int(config["batch_size"])

    def predict(x):
        network.eval()
        output = []
        with torch.no_grad():
            for start in range(0, len(x), batch_size):
                batch = torch.as_tensor(x[start:start+batch_size], device=dev, dtype=torch.float32)
                output.append(network(batch).reshape(-1).cpu().numpy())
        return np.concatenate(output)

    _model_sync(dev)
    started = time.perf_counter()
    best_score, best_state, best_epoch, stale = -np.inf, None, 0, 0
    history = []
    for epoch in range(1, int(config["max_epochs"]) + 1):
        network.train()
        permutation = torch.randperm(len(xtrain), generator=permutation_generator)
        loss_sum = 0.0
        for start in range(0, len(xtrain), batch_size):
            indices = permutation[start:start+batch_size]
            xb, yb = xtrain[indices].to(dev), ytrain[indices].to(dev)
            optimizer.zero_grad(set_to_none=True)
            logits = network(xb).reshape(-1)
            loss = criterion(logits, yb)
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite MLP training loss.")
            loss.backward()
            optimizer.step()
            loss_sum += float(loss.detach().item()) * len(indices)
        val_metrics = _model_metrics(predict(arrays[1]), labels[1])
        score = val_metrics["balanced_accuracy"]
        history.append({"epoch": epoch, "train_loss": loss_sum / len(xtrain),
                        "validation_balanced_accuracy": score})
        if score > best_score:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {name: value.detach().cpu().clone()
                          for name, value in network.state_dict().items()}
        else:
            stale += 1
        if stale >= int(config["patience"]):
            break
    network.load_state_dict(best_state)
    # Test records are evaluated exactly once, after the validation selection.
    result = _model_metrics(predict(arrays[2]), labels[2])
    _model_sync(dev)
    result.update({"val_best_balanced_accuracy": float(best_score),
                   "best_epoch": int(best_epoch), "epochs": int(epoch),
                   "train_seconds": time.perf_counter() - started,
                   "training_seed": int(seed), "decision_threshold_logit": 0.0,
                   "architecture": [h, *hidden, 1], "training_history": history,
                   "stopped_early": bool(epoch < int(config["max_epochs"]))})
    del network, optimizer, best_state
    return result


class FaceAuditor:
    """Existing frozen VGGFace2 InceptionResnetV1 cross-image identity auditor."""

    def __init__(self, device="cuda", torchcache=None, batch_size=64):
        if torchcache is not None:
            Path(torchcache).mkdir(parents=True, exist_ok=True)
            os.environ["TORCH_HOME"] = str(torchcache)
        from facenet_pytorch import InceptionResnetV1
        self.device = torch.device(device)
        self.batch_size = int(batch_size)
        self.model = InceptionResnetV1(pretrained="vggface2").eval().to(self.device)
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)
        digest = hashlib.sha256()
        for name, value in sorted(self.model.state_dict().items()):
            digest.update(name.encode())
            digest.update(value.detach().cpu().contiguous().numpy().tobytes())
        self.metadata = {
            "architecture": "InceptionResnetV1", "pretrained": "vggface2",
            "frozen": True, "state_sha256": digest.hexdigest(),
            "preprocessing": "RGB [0,1], bilinear 160x160, (255*x-127.5)/128",
            "comparison": "cosine similarity to clean cross-image references",
        }

    @torch.no_grad()
    def encode_gallery(self, images):
        """Accept uint8 or [0,1] float RGB, NCHW or NHWC, return n x 512 numpy."""
        tensor = torch.as_tensor(images)
        if tensor.ndim != 4:
            raise ValueError("Expected a four-dimensional RGB image batch.")
        if tensor.shape[1] != 3 and tensor.shape[-1] == 3:
            tensor = tensor.permute(0, 3, 1, 2)
        if tensor.shape[1] != 3:
            raise ValueError("Auditor images must have three RGB channels.")
        uint8 = tensor.dtype == torch.uint8
        embeddings = []
        for start in range(0, len(tensor), self.batch_size):
            batch = tensor[start:start+self.batch_size].to(self.device, dtype=torch.float32)
            if uint8:
                batch = batch / 255.0
            if not torch.isfinite(batch).all() or float(batch.min()) < -1e-6 or float(batch.max()) > 1.000001:
                raise ValueError("Auditor floating images must be finite and in [0,1].")
            batch = torch_functional.interpolate(batch, size=(160, 160), mode="bilinear", align_corners=False)
            batch = (batch * 255.0 - 127.5) / 128.0
            embeddings.append(torch_functional.normalize(self.model(batch), dim=1).cpu())
        if not embeddings:
            return np.empty((0, 512), dtype=np.float32)
        return torch.cat(embeddings).numpy()

    @torch.no_grad()
    def linkage(self, qhat, F, gallery_embeddings, lineup_indices, correct_pos):
        """Decode an orthonormal public-SVD representation and score fixed lineups.

        qhat: n_query x h; F: (3*64*64) x h, using CHW flattening.
        lineups: n_query x n_repeated_galleries x n_candidates (or n_query x k).
        Returns identity-balanced linkage when each query represents one identity.
        """
        query = _model_numpy(qhat, np.float32)
        basis = torch.as_tensor(F, dtype=torch.float32, device=self.device)
        if basis.shape[0] != 3 * 64 * 64 or basis.shape[1] != query.shape[1]:
            raise ValueError("F and qhat must use the public RGB64 SVD projection.")
        query_embeddings = []
        for start in range(0, len(query), self.batch_size):
            reduced = torch.as_tensor(query[start:start+self.batch_size], device=self.device)
            decoded = (reduced @ basis.T).clamp(0.0, 1.0).reshape(-1, 3, 64, 64)
            query_embeddings.append(self.encode_gallery(decoded))
        query_embeddings = np.concatenate(query_embeddings)
        gallery = _model_numpy(gallery_embeddings, np.float32)
        indices = np.asarray(lineup_indices, dtype=np.int64)
        truth = np.asarray(correct_pos, dtype=np.int64)
        if indices.ndim == 2:
            indices, truth = indices[:, None, :], truth.reshape(-1, 1)
        if indices.ndim != 3 or len(indices) != len(query) or truth.shape != indices.shape[:2]:
            raise ValueError("Lineup indices and correct positions do not match query count.")
        if (indices < 0).any() or (indices >= len(gallery)).any():
            raise ValueError("Gallery index outside available reference embeddings.")
        if (truth < 0).any() or (truth >= indices.shape[-1]).any():
            raise ValueError("Correct position outside candidate lineup.")
        scores = np.einsum("qd,qgkd->qgk", query_embeddings, gallery[indices])
        success = scores.argmax(axis=-1) == truth
        return {"linkage_rate": float(success.mean()),
                "per_query_success": success.mean(axis=1).tolist(),
                "n_queries": int(len(query)), "lineups_per_query": int(indices.shape[1]),
                "n_candidates": int(indices.shape[2]), "n_lineups": int(success.size),
                "chance_rate": 1.0 / indices.shape[2]}


def protocol_self_test():
    """Small CPU algebra and reference-gauge checks, without pretrained weights."""
    bank = generate_bank(173, pmax=3, h=5, r=17)
    short = generate_bank(173, pmax=2, h=5, r=17)
    np.testing.assert_array_equal(bank["O"][:2], short["O"])
    np.testing.assert_array_equal(bank["A"], short["A"])
    np.testing.assert_allclose(bank["A"].mean(0), 0, atol=2e-15)
    np.testing.assert_allclose(bank["A"].T @ bank["A"], np.eye(5), atol=2e-14)
    data = [np.random.default_rng(i).normal(size=(11, 5)) for i in range(3)]
    uploads = upload_blocks(data, bank)
    clean = noisy_anchor_bank(bank, 173, 0, 0, 0.0)
    maps, diag = align_anchors(clean, device="cpu", tol=1e-10, max_iters=100)
    aligned = apply_maps(uploads, maps)
    for i, result in enumerate(aligned):
        np.testing.assert_allclose(result, data[i] @ bank["O"][0] + bank["psi"][0], atol=1e-11)
        recovered = op_reconstruct(bank["A"], clean[i], uploads[i], device="cpu")
        np.testing.assert_allclose(recovered, data[i], atol=1e-11)
    # Coupling isotropic private noise by orthogonal rotation yields the exact
    # common-reference noisy matrix asserted by the equivalence result.
    errors = [0.1 * np.random.default_rng(500+i).normal(size=q.shape)
              for i, q in enumerate(data)]
    noisy_uploads = [y + errors[i] @ bank["O"][i] for i, y in enumerate(uploads)]
    noisy_aligned = apply_maps(noisy_uploads, maps)
    for i, result in enumerate(noisy_aligned):
        common = (data[i] + errors[i]) @ bank["O"][0] + bank["psi"][0]
        np.testing.assert_allclose(result, common, atol=1e-11)
    np.testing.assert_array_equal(maps["rotations_R"][0], np.eye(5))
    # Common right multiplication must disappear in reference normalization.
    gauge = _model_haar(np.random.default_rng(77), 5)
    raw = maps["rotations_R"] @ gauge
    normalized = raw @ raw[0].T
    np.testing.assert_allclose(normalized, maps["rotations_R"], atol=1e-13)
    noise1 = noisy_anchor_bank(bank, 173, 3, 2, 0.024)
    noise2 = noisy_anchor_bank(short, 173, 3, 2, 0.024)
    for a, b in zip(noise1, noise2):
        np.testing.assert_array_equal(a, b)
    # Exercise the r-square spectral route too (p*h > r).
    wide = generate_bank(31, pmax=5, h=5, r=17)
    _, wide_diag = align_anchors(noisy_anchor_bank(wide, 31, 0, 0, 0), device="cpu",
                                tol=1e-10, max_iters=100)
    assert diag["converged"] and wide_diag["converged"]
    return {"passed": True, "noiseless_gopp_relative_residual": diag["relative_gopp_residual"],
            "wide_spectral_relative_residual": wide_diag["relative_gopp_residual"]}


In [ ]:
"""Replay ten noise levels from saved seed-0 deployments; no model fitting."""
import gc
import hashlib
import json
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJECT = Path('/content/drive/MyDrive/Noisy Anchor Alignment')
OUTPUT = PROJECT/'results'/'figures'/'manuscript_inputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(2)

def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

def export_gallery(dataset, entry, bank, scales, sigmas, participant_count, raw_path):
    path = Path(entry['data_path'])
    assert sha(path) == entry['data_sha256'], 'Deployment checksum mismatch'
    data = torch.load(path, map_location='cpu', weights_only=True)
    F = data['F'].numpy()
    assert hashlib.sha256(F.tobytes()).hexdigest() == entry['projection_sha256']
    assert len(scales) == len(sigmas) == 10
    np.testing.assert_allclose(scales, np.arange(1, 11)*.006)
    np.testing.assert_allclose(sigmas, np.arange(1, 11)*.3)
    target, seed, realization = 1, 0, 0
    query = data['query_q'].numpy().astype(np.float64)
    assert len(query) == 100 and len(set(data['query_identities'][:10])) == 10
    original = data['query_images'].numpy()[:10]
    def decode(q):
        return np.clip(q.astype(np.float32) @ F.T, 0, 1).reshape(-1, 3, 64, 64)
    zero = decode(query[:10])
    upload = query @ bank['O'][target] + bank['psi'][target]
    images = np.empty((4, 10, 3, 64, 64), np.float32)
    rates = np.full((4, 10), np.nan)
    raw = pd.read_csv(raw_path)
    raw = raw[(raw.outer_seed == seed) & (raw.participant_count == participant_count) & (raw.realization == realization)]
    def recorded_rate(method, level, scale):
        rows = raw[(raw.method == method) & (raw.level_index == level)]
        assert len(rows) == 1, (dataset, method, level, len(rows))
        row = rows.iloc[0]
        assert abs(float(row.noise_scale)-scale) < 1e-10
        # Both exports must refer to the deployment used by the saved score.
        assert str(row.deployment_hash) == str(entry['config_hash'])
        return float(row.linkage_rate), str(row.run_id)
    zero_rate, zero_id = recorded_rate('C-GDP', 0, 0)
    conditions = []
    for column, (v, sigma) in enumerate(zip(scales, sigmas)):
        level = column + 1
        noise = np.random.default_rng(_model_seed(seed, 'anchor-noise', level, realization, target)).standard_normal(bank['A'].shape)
        B = bank['A'] @ bank['O'][target] + bank['psi'][target] + float(v)*noise
        recovered = op_reconstruct(bank['A'], B, upload, device='cpu')
        anchor = decode(recovered[column:column+1])[0]
        location = data['query_locations'][column]
        split = location['split']
        private_noise = np.random.default_rng(_model_seed(seed, 'private-noise', level, realization, split, target)).standard_normal(tuple(data['participants'][target][split+'_q'].shape))
        q_private = query[column:column+1] + float(sigma)*private_noise[location['position']:location['position']+1] @ bank['O'][0].T
        private = decode(q_private)[0]
        images[:, column] = np.stack([original[column], zero[column], private, anchor])
        anchor_rate, anchor_id = recorded_rate('NAA-GDP', level, float(v))
        private_rate, private_id = recorded_rate('C-GDP-private-noise', level, float(sigma))
        rates[1:, column] = [zero_rate, private_rate, anchor_rate]
        conditions.append({'column':column+1, 'query_index':column, 'v':float(v), 'sigma':float(sigma), 'level_index':level, 'realization':realization, 'anchor_run_id':anchor_id, 'private_run_id':private_id, 'anchor_linkage_rate':anchor_rate, 'private_linkage_rate':private_rate})
    assert np.isfinite(images).all() and images.min() >= 0 and images.max() <= 1
    np.savez_compressed(OUTPUT/f'{dataset}_gallery.npz', images=images, linkage_rates=rates, anchor_scales=scales, private_sigmas=sigmas)
    manifest = {'dataset':dataset, 'seed':seed, 'participant_count':participant_count, 'target_participant':target, 'query_indices':list(range(10)), 'query_identities':list(data['query_identities'][:10]), 'selection':'First ten distinct queries in saved deployment order. Each column preserves one source face across methods; columns use increasing positive noise levels. No quality-based selection.', 'conditions':conditions, 'projection_sha256':entry['projection_sha256'], 'data_sha256':entry['data_sha256'], 'deployment_hash':entry['config_hash'], 'zero_run_id':zero_id, 'zero_linkage_rate':zero_rate, 'score_source':str(raw_path), 'linkage_annotation':'Saved seed-0 realization-0 linkage across all 1000 ten-way lineups for that condition, not a per-image score or a five-seed mean.', 'scope':'Visual-only replay of saved deployment and exact level-indexed releases. No utility fitting or modification of experiment outputs.'}
    (OUTPUT/f'{dataset}_gallery_manifest.json').write_text(json.dumps(manifest, indent=2))
    print(f'{dataset}: ten-column gallery exported; all 21 scores matched to saved deployment {entry["config_hash"]}', flush=True)
    del data, images, private_noise
    gc.collect()

celeba = PROJECT/'results'/'celeba_parallel'
canonical = json.loads((celeba/'shared'/'canonical.json').read_text())
entry = canonical['deployments']['0']
assert sha(entry['bank_path']) == entry['bank_sha256']
with np.load(entry['bank_path'], allow_pickle=False) as saved:
    bank = {key:saved[key] for key in ('A','O','psi')}
export_gallery('celeba', entry, bank, canonical['scientific_config']['anchor_scales'], canonical['scientific_config']['private_sigmas'], 10, celeba/'merged'/'raw_results.csv')
del bank

lfwa_full = PROJECT/'results'/'lfwa_full_sweep'
lfwa_protocol = json.loads((lfwa_full/'protocol.json').read_text())
entry = json.loads((lfwa_full/'deployments'/'seed_0'/'deployment_manifest.json').read_text())
bank = generate_bank(0, 5, 400, 1600)
export_gallery('lfwa', entry, bank, lfwa_protocol['config']['anchor_scales'], lfwa_protocol['config']['private_sigmas'], 5, lfwa_full/'raw_results.csv')
del bank

# Quantitative plots use the completed CelebA runs and completed LFWA full sweep.
diagnostics = {}
for folder in [PROJECT/'results'/'celeba'/'records', *[celeba/'jobs'/job/'records' for job in ('p50','p2_20','p5_10')]]:
    for path in folder.glob('*_p10_NAA-GDP_*.json'):
        row = json.loads(path.read_text())
        assert row['run_id'] not in diagnostics
        a = row['alignment']
        diagnostics[row['run_id']] = {**{k:row[k] for k in ('run_id','outer_seed','participant_count','noise_scale','level_index','realization')}, **{k:a[k] for k in ('iterations','converged','last_step','fixed_point_residual','elapsed_seconds','initialization_seconds','iteration_seconds')}}
assert len(diagnostics) == 200
pd.DataFrame(diagnostics.values()).sort_values(['outer_seed','level_index','realization']).to_csv(OUTPUT/'celeba_gpm_diagnostics.csv', index=False)
counts = []
lfwa = PROJECT/'results'/'lfwa_full_sweep'
for dataset, paths in [('CelebA',[celeba/'shared'/'deployments'/f'seed_{s}'/'deployment_manifest.json' for s in range(5)]), ('LFWA',[lfwa/'deployments'/f'seed_{s}'/'deployment_manifest.json' for s in range(5)])]:
    for seed, path in enumerate(paths):
        manifest = json.loads(path.read_text())
        counts.extend([{'dataset':dataset, 'outer_seed':seed, **x} for x in manifest['sample_counts']])
pd.DataFrame(counts).to_csv(OUTPUT/'sample_counts.csv', index=False)
gallery_zip = OUTPUT.parent/'reconstruction_gallery_inputs.zip'
import zipfile
with zipfile.ZipFile(gallery_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for dataset in ('lfwa','celeba'):
        for suffix in ('gallery.npz','gallery_manifest.json'):
            path = OUTPUT/f'{dataset}_{suffix}'
            archive.write(path, path.name)
print('COMPLETE: both ten-level reconstruction galleries exported independently of utility training.', flush=True)


In [ ]:
"""Render the seven manuscript figures from completed runs, without fitting models."""
from pathlib import Path
import argparse, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

NAA='NAA-GDP'; PRIVATE='C-GDP-private-noise'
COLORS={NAA:'#947389',PRIVATE:'#536C8A','I-GDP':'#73777F','Local-only':'#AA895C',
        'MP':'#536C8A','AM':'#58988F','OP':'#947389'}
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8,'axes.labelsize':8,
 'legend.fontsize':7,'xtick.labelsize':7,'ytick.labelsize':7,'axes.titlesize':8,
 'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,
 'lines.linewidth':1.25,'lines.markersize':3.3,'pdf.fonttype':42,'ps.fonttype':42,
 'svg.fonttype':'none','savefig.facecolor':'white'})

def stats(df,keys,metrics):
    seed=df.groupby(keys+['outer_seed'])[metrics].mean().reset_index()
    assert seed.groupby(keys).size().eq(5).all()
    result=seed.groupby(keys)[metrics].agg(['mean','std'])
    result.columns=['_'.join(c) for c in result.columns]
    return result.reset_index()

def summary(df):
    s=stats(df,['method','participant_count','noise_scale'],['balanced_accuracy','linkage_rate'])
    for c in s:
        if c.endswith(('_mean','_std')): s[c]*=100
    return s

def polished(ax,axis='both'):
    ax.grid(True,axis=axis,color='#DADDE0',lw=.5,alpha=.65);ax.set_axisbelow(True)

def series(u,method,p):
    base='AA-GDP' if method==NAA else 'C-GDP'
    return u[(u.participant_count==p)&u.method.isin([base,method])].sort_values('noise_scale')

def save(fig,name,out):
    for ext in ('pdf','png'):
        fig.savefig(out/f'{name}.{ext}',dpi=200,bbox_inches='tight',pad_inches=.03)
    plt.close(fig)

def tradeoff(u,p,dataset,out):
    fig,ax=plt.subplots(figsize=(3.45,2.75))
    for method,marker,label in [(NAA,'o','Anchor noise (NAA-GDP)'),(PRIVATE,'s','Private noise (C-GDP)')]:
        d=series(u,method,p)
        ax.errorbar(d.linkage_rate_mean,d.balanced_accuracy_mean,
            xerr=d.linkage_rate_std,yerr=d.balanced_accuracy_std,
            color=COLORS[method],marker=marker,capsize=1.6,elinewidth=.55,capthick=.55,label=label)
    for m,ls in [('I-GDP','--'),('Local-only',':')]:
        y=u[(u.method==m)&(u.participant_count==p)].balanced_accuracy_mean.iloc[0]
        ax.axhline(y,color=COLORS[m],ls=ls,lw=1,label=m)
    ax.axvline(10,color='#B5B8BB',ls='--',lw=.65,zorder=0)
    ax.set(xlabel='Identity linkage (%)',ylabel='Balanced accuracy (%)',
           xlim=(7,81 if dataset=='CelebA' else 73),ylim=(60,92) if dataset=='CelebA' else (48,79))
    if dataset=='LFWA':
        plotted=pd.concat([series(u,m,p) for m in (NAA,PRIVATE)])
        assert (plotted.linkage_rate_mean-plotted.linkage_rate_std).min()>=ax.get_xlim()[0]
        assert (plotted.linkage_rate_mean+plotted.linkage_rate_std).max()<=ax.get_xlim()[1]
        assert (plotted.balanced_accuracy_mean-plotted.balanced_accuracy_std).min()>=ax.get_ylim()[0]
        assert (plotted.balanced_accuracy_mean+plotted.balanced_accuracy_std).max()<=ax.get_ylim()[1]
    polished(ax)
    h,l=ax.get_legend_handles_labels()
    # Utility references are intentionally not assigned a privacy coordinate.
    order=[2,3,0,1] if l[0]=='I-GDP' else list(range(len(l)))
    fig.legend([h[i] for i in order],[l[i] for i in order],loc='lower center',
               ncol=1,frameon=False,bbox_to_anchor=(.57,-.012),fontsize=7)
    fig.subplots_adjust(left=.17,right=.98,top=.97,bottom=.40)
    save(fig,dataset.lower()+'_privacy_utility',out)

def participants(u,out):
    fig,ax=plt.subplots(figsize=(3.45,2.8))
    colors=['#536C8A','#58988F','#AA895C','#947389','#465366']
    handles=[]
    for p,color,marker in zip([2,5,10,20,50],colors,['o','s','^','D','v']):
        d=series(u,NAA,p).sort_values('linkage_rate_mean')
        x=d.linkage_rate_mean.to_numpy();y=d.balanced_accuracy_mean.to_numpy();sd=d.balanced_accuracy_std.to_numpy()
        ax.plot(x,y,color=color,marker=marker,markersize=2.6,label=f'p = {p}')
        ax.fill_between(x,y-sd,y+sd,color=color,alpha=.09,lw=0)
        y0=u[(u.method=='I-GDP')&(u.participant_count==p)].balanced_accuracy_mean.iloc[0]
        ax.axhline(y0,color=color,lw=.8,ls='--',alpha=.85)
        handles.append(Line2D([],[],color=color,marker=marker,label=f'p = {p}'))
    ax.set(xlabel='Identity linkage (%)',ylabel='Balanced accuracy (%)',xlim=(8,80),ylim=(77,91))
    polished(ax)
    fig.legend(handles=handles,loc='lower center',ncol=3,frameon=False,bbox_to_anchor=(.57,.01))
    fig.subplots_adjust(left=.17,right=.98,top=.98,bottom=.29)
    save(fig,'celeba_participant_sensitivity',out)

def lfwa_bars(raw,attacks,out):
    a=stats(attacks[attacks.method==NAA],['noise_scale','attack'],['linkage_rate'])
    levels=np.sort(raw.loc[raw.method==NAA,'noise_scale'].unique())
    assert len(levels)==10 and levels.min()>0
    fig,ax=plt.subplots(figsize=(6.9,2.5));xx=np.arange(len(levels));width=.23
    for j,attack in enumerate(['MP','AM','OP']):
        d=a[a.attack==attack].sort_values('noise_scale')
        np.testing.assert_allclose(d.noise_scale,levels,rtol=0,atol=1e-12)
        ax.bar(xx+(j-1)*width,d.linkage_rate_mean*100,width=.215,
          yerr=d.linkage_rate_std*100,color=COLORS[attack],label=attack,
          capsize=2,error_kw={'elinewidth':.7,'capthick':.7})
    ax.axhline(10,color='#8B9095',ls='--',lw=.8)
    ax.set_xticks(xx,[f'{v:.3f}' for v in levels])
    upper=max(65,5*np.ceil(100*(a.linkage_rate_mean+a.linkage_rate_std).max()/5))
    ax.set(xlabel='Anchor-noise scale, $v$',ylabel='Identity linkage (%)',ylim=(0,upper))
    ax.legend(frameon=False,ncol=3,loc='upper right')
    polished(ax,'y');fig.subplots_adjust(left=.08,right=.995,top=.97,bottom=.21)
    save(fig,'lfwa_anchor_noise_identity_leakage',out)

def celeba_bars(u,out):
    fig,axes=plt.subplots(1,2,figsize=(6.9,2.2),sharey=True)
    for ax,m,title in zip(axes,[NAA,PRIVATE],['(a) Anchor noise: OP','(b) Private noise: common-transform inversion']):
        d=series(u,m,10);xx=np.arange(len(d))
        ax.bar(xx,d.linkage_rate_mean,yerr=d.linkage_rate_std,color=COLORS[m],width=.72,
             capsize=1.5,error_kw={'elinewidth':.55,'capthick':.55})
        ax.axhline(10,color='#8B9095',ls='--',lw=.8)
        labels=[('0' if v==0 else f'{v:.3f}') if m==NAA else f'{v:g}' for v in d.noise_scale]
        ax.set_xticks(xx,labels,rotation=45,ha='right')
        ax.set(xlabel='Anchor-noise scale, $v$' if m==NAA else r'Private-data noise SD, $\sigma$',ylim=(0,82))
        ax.set_title(title,fontsize=7.7);polished(ax,'y')
    axes[0].set_ylabel('Identity linkage (%)')
    fig.subplots_adjust(left=.08,right=.99,top=.85,bottom=.30,wspace=.13)
    save(fig,'celeba_noise_identity_leakage',out)

"""One two-panel, ten-column reconstruction gallery for the main manuscript."""
def render_combined_gallery(inputs, out):
    # Geometry is in inches so each face stays square at manuscript scale.
    width, height = 7.10, 6.12
    left, face, gap = 1.06, .572, .026
    top, panel_height, panel_gap = .32, 2.77, .27
    row_y = [0, .63, 1.34, 2.04]
    labels = ['Original', 'C-GDP\n(projection only)', 'C-GDP\n(private noise)', 'NAA-GDP']
    fig = plt.figure(figsize=(width, height), facecolor='white')
    for panel, dataset in enumerate(('lfwa','celeba')):
        archive = np.load(inputs/f'{dataset}_gallery.npz', allow_pickle=False)
        images, rates = archive['images'], archive['linkage_rates']
        assert images.shape == (4,10,3,64,64)
        offset = top + panel*(panel_height+panel_gap)
        panel_title = '(a) LFWA' if panel == 0 else '(b) CelebA'
        fig.text(left/width, 1-(offset-.15)/height, panel_title, ha='left', va='bottom', fontsize=9, fontweight='semibold')
        for row in range(4):
            y_top = offset+row_y[row]
            fig.text((left-.075)/width, 1-(y_top+face/2)/height, labels[row], ha='right', va='center', fontsize=7.5, linespacing=1.2)
            for column in range(10):
                x = left+column*(face+gap)
                ax = fig.add_axes([x/width, 1-(y_top+face)/height, face/width, face/height])
                ax.imshow(np.moveaxis(images[row,column],0,-1), interpolation='nearest')
                ax.axis('off')
                if row:
                    fig.text((x+face/2)/width, 1-(y_top+face+.022)/height, f'{100*rates[row,column]:.1f}%', ha='center', va='top', fontsize=7.4)
    # The figure caption explains that percentages cover all 1,000 lineups.
    out.mkdir(parents=True, exist_ok=True)
    for extension in ('pdf','png'):
        fig.savefig(out/f'face_reconstruction_gallery.{extension}', dpi=230, bbox_inches='tight', pad_inches=.025)
    plt.close(fig)
    print('Rendered one combined LFWA/CelebA ten-column gallery.', flush=True)


def gpm(inputs,raw,out):
    g=pd.read_csv(inputs/'celeba_gpm_diagnostics.csv')
    target=raw[(raw.method==NAA)&(raw.participant_count==10)]
    assert len(g)==200 and set(g.run_id)==set(target.run_id)
    assert int(g.converged.sum())==120
    fig,axes=plt.subplots(1,3,figsize=(6.9,2.35))
    for ax,field,title,ylabel,log in zip(axes,
       ['iterations','fixed_point_residual','elapsed_seconds'],
       ['(a) Iterations','(b) Relative fixed-point residual','(c) Alignment time'],
       ['GPM iterations','Relative residual','Seconds'],[False,True,True]):
        for converged,color,label in [(True,'#58988F','Converged (120)'),(False,'#947389','Capped (80)')]:
            d=g[g.converged==converged]
            # Small deterministic offsets separate the 20 observations at each tested level.
            offsets=(d.outer_seed.to_numpy()*4+d.realization.to_numpy()-9.5)*.000105
            ax.scatter(d.noise_scale+offsets,d[field],s=10,edgecolors=color,
              facecolors=color if converged else 'none',linewidths=.5,alpha=.8,label=label)
        ax.axvline(.0530,color='#9D9FA1',ls='--',lw=.75)
        ax.set(xlabel='Anchor-noise scale, $v$',ylabel=ylabel,xlim=(0,.064))
        ax.set_xticks([0,.02,.04,.06],['0','.02','.04','.06']);ax.set_title(title,fontsize=7.5)
        if log: ax.set_yscale('log')
        polished(ax)
    handles,labels=axes[0].get_legend_handles_labels()
    handles.append(Line2D([],[],color='#9D9FA1',ls='--',label='$v_{PT}=0.0530$'));labels.append('$v_{PT}=0.0530$')
    fig.legend(handles,labels,ncol=3,frameon=False,loc='lower center',bbox_to_anchor=(.53,0))
    fig.subplots_adjust(left=.075,right=.985,bottom=.31,top=.86,wspace=.52)
    save(fig,'celeba_gpm_diagnostics',out)

def main(data,inputs,out):
    out.mkdir(parents=True,exist_ok=True)
    c=pd.read_csv(data/'celeba_raw_results.csv');l=pd.read_csv(data/'lfwa_raw_results.csv')
    assert len(c)==1300 and len(l)==420 and c.run_id.nunique()==1300 and l.run_id.nunique()==420
    for df in [c,l]:
        assert set(df.outer_seed)==set(range(5))
        assert df[df.method.isin(['I-GDP','Local-only'])].linkage_rate.isna().all()
        for k,rows in df.groupby(['outer_seed','participant_count','method','noise_scale']):
            assert len(rows)==(4 if k[-1]>0 else 1)
    cs=summary(c);ls=summary(l)
    cs.to_csv(out/'celeba_plot_statistics.csv',index=False);ls.to_csv(out/'lfwa_plot_statistics.csv',index=False)
    tradeoff(cs,10,'CelebA',out);tradeoff(ls,5,'LFWA',out);participants(cs,out)
    lfwa_bars(l,pd.read_csv(data/'lfwa_attack_results.csv'),out);celeba_bars(cs,out)
    render_combined_gallery(inputs,out);gpm(inputs,c,out)
    print('Rendered 7 manuscript figures from complete, validated inputs.')


import tempfile,shutil
from IPython.display import display,Image
with tempfile.TemporaryDirectory(prefix='naa_manuscript_figures_') as temp:
    data=Path(temp)
    for source,name in [
        (PROJECT/'results'/'celeba_parallel'/'merged'/'raw_results.csv','celeba_raw_results.csv'),
        (PROJECT/'results'/'lfwa_full_sweep'/'raw_results.csv','lfwa_raw_results.csv'),
        (PROJECT/'results'/'lfwa_full_sweep'/'attack_results.csv','lfwa_attack_results.csv')]:
        shutil.copyfile(source,data/name)
    main(data,PROJECT/'results'/'figures'/'manuscript_inputs',PROJECT/'results'/'figures'/'manuscript')
print('Manuscript PDFs and PNGs saved in results/figures/manuscript.')


In [ ]:
# Figure 3: separate LFWA and CelebA subfigures from the saved gallery inputs.
from pathlib import Path
import hashlib
import json
import zipfile
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image, FileLink

PROJECT = Path("/content/drive/MyDrive/Noisy Anchor Alignment")
FIGURE3_INPUTS = PROJECT / "results" / "figures" / "manuscript_inputs"
FIGURE3_OUTPUT = PROJECT / "results" / "figures" / "manuscript"
FIGURE3_OUTPUT.mkdir(parents=True, exist_ok=True)

def render_figure3_subfigures(inputs, out, datasets=("lfwa", "celeba"), show=True):
    labels = ["Original", "C-GDP\n(projection only)",
              "C-GDP\n(private noise)", "NAA-GDP"]
    width, height = 7.10, 3.08
    left, face, gap, top = 1.06, .572, .026, .32
    row_y = [0, .63, 1.34, 2.04]
    outputs = []
    manifest = out / "figure3_subfigures_manifest.json"
    provenance = json.loads(manifest.read_text()) if manifest.exists() else {}
    style = {"font.family": "DejaVu Sans", "font.size": 8,
             "pdf.fonttype": 42, "ps.fonttype": 42,
             "savefig.facecolor": "white"}
    with plt.rc_context(style):
        for dataset in datasets:
            source = inputs / f"{dataset}_gallery.npz"
            source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
            with np.load(source, allow_pickle=False) as archive:
                images = archive["images"].copy()
                rates = archive["linkage_rates"].copy()
            assert images.shape == (4, 10, 3, 64, 64)
            assert rates.shape == (4, 10)
            assert np.isfinite(images).all()
            assert np.isfinite(rates[1:]).all()
            fig = plt.figure(figsize=(width, height), facecolor="white")
            for row in range(4):
                y_top = top + row_y[row]
                fig.text((left-.075)/width, 1-(y_top+face/2)/height,
                         labels[row], ha="right", va="center",
                         fontsize=7.5, linespacing=1.2)
                for column in range(10):
                    x = left + column*(face+gap)
                    ax = fig.add_axes([x/width, 1-(y_top+face)/height,
                                       face/width, face/height])
                    ax.imshow(np.moveaxis(images[row, column], 0, -1),
                              interpolation="nearest")
                    ax.axis("off")
                    if row:
                        fig.text((x+face/2)/width,
                                 1-(y_top+face+.022)/height,
                                 f"{100*rates[row, column]:.1f}%",
                                 ha="center", va="top", fontsize=7.4)
            for extension in ("pdf", "png"):
                target = out / f"face_reconstruction_gallery_{dataset}.{extension}"
                fig.savefig(target, dpi=230, bbox_inches="tight",
                            pad_inches=.025)
                outputs.append(target)
            plt.close(fig)
            assert hashlib.sha256(source.read_bytes()).hexdigest() == source_hash
            provenance[dataset] = {
                "input": str(source), "sha256": source_hash,
                "deployment_seed": 0, "noise_realization": 0,
                "images": 40, "displayed_linkage_rates": rates[1:].tolist(),
                "subcaption": "(a) LFWA" if dataset == "lfwa" else "(b) CelebA"}
            print(f"{dataset.upper()}: 40 saved images, 30 unchanged linkage annotations.")
            if show:
                display(Image(filename=str(out / f"face_reconstruction_gallery_{dataset}.png")))
    manifest = out / "figure3_subfigures_manifest.json"
    manifest.write_text(json.dumps(provenance, indent=2))
    outputs.append(manifest)
    return outputs

figure3_files = render_figure3_subfigures(FIGURE3_INPUTS, FIGURE3_OUTPUT)
figure3_zip = FIGURE3_OUTPUT / "figure3_subfigures.zip"
with zipfile.ZipFile(figure3_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for target in figure3_files:
        archive.write(target, target.name)
print(f"Saved Figure 3 subfigures: {figure3_zip}")
import shutil
shutil.copyfile(figure3_zip, "/content/figure3_subfigures.zip")
print("Download figure3_subfigures.zip from the Colab Files panel.")


## Refresh LFWA figures only

Run after mounting Drive. This standalone cell reads the completed full sweep and saved gallery, checks their provenance and uncertainty aggregation, and refreshes only LFWA figures. It saves a ZIP containing the figures, statistics, and validation manifest. No model fitting or CelebA rendering.


In [ ]:
# Refresh LFWA figures only, from completed saved results. No model fitting.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display, Image, FileLink

"""Render the seven manuscript figures from completed runs, without fitting models."""
from pathlib import Path
import argparse, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

NAA='NAA-GDP'; PRIVATE='C-GDP-private-noise'
COLORS={NAA:'#947389',PRIVATE:'#536C8A','I-GDP':'#73777F','Local-only':'#AA895C',
        'MP':'#536C8A','AM':'#58988F','OP':'#947389'}
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8,'axes.labelsize':8,
 'legend.fontsize':7,'xtick.labelsize':7,'ytick.labelsize':7,'axes.titlesize':8,
 'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,
 'lines.linewidth':1.25,'lines.markersize':3.3,'pdf.fonttype':42,'ps.fonttype':42,
 'svg.fonttype':'none','savefig.facecolor':'white'})

def stats(df,keys,metrics):
    seed=df.groupby(keys+['outer_seed'])[metrics].mean().reset_index()
    assert seed.groupby(keys).size().eq(5).all()
    result=seed.groupby(keys)[metrics].agg(['mean','std'])
    result.columns=['_'.join(c) for c in result.columns]
    return result.reset_index()

def summary(df):
    s=stats(df,['method','participant_count','noise_scale'],['balanced_accuracy','linkage_rate'])
    for c in s:
        if c.endswith(('_mean','_std')): s[c]*=100
    return s

def polished(ax,axis='both'):
    ax.grid(True,axis=axis,color='#DADDE0',lw=.5,alpha=.65);ax.set_axisbelow(True)

def series(u,method,p):
    base='AA-GDP' if method==NAA else 'C-GDP'
    return u[(u.participant_count==p)&u.method.isin([base,method])].sort_values('noise_scale')

def save(fig,name,out):
    for ext in ('pdf','png'):
        fig.savefig(out/f'{name}.{ext}',dpi=200,bbox_inches='tight',pad_inches=.03)
    plt.close(fig)

def tradeoff(u,p,dataset,out):
    fig,ax=plt.subplots(figsize=(3.45,2.75))
    for method,marker,label in [(NAA,'o','Anchor noise (NAA-GDP)'),(PRIVATE,'s','Private noise (C-GDP)')]:
        d=series(u,method,p)
        ax.errorbar(d.linkage_rate_mean,d.balanced_accuracy_mean,
            xerr=d.linkage_rate_std,yerr=d.balanced_accuracy_std,
            color=COLORS[method],marker=marker,capsize=1.6,elinewidth=.55,capthick=.55,label=label)
    for m,ls in [('I-GDP','--'),('Local-only',':')]:
        y=u[(u.method==m)&(u.participant_count==p)].balanced_accuracy_mean.iloc[0]
        ax.axhline(y,color=COLORS[m],ls=ls,lw=1,label=m)
    ax.axvline(10,color='#B5B8BB',ls='--',lw=.65,zorder=0)
    ax.set(xlabel='Identity linkage (%)',ylabel='Balanced accuracy (%)',
           xlim=(7,81 if dataset=='CelebA' else 73),ylim=(60,92) if dataset=='CelebA' else (48,79))
    if dataset=='LFWA':
        plotted=pd.concat([series(u,m,p) for m in (NAA,PRIVATE)])
        assert (plotted.linkage_rate_mean-plotted.linkage_rate_std).min()>=ax.get_xlim()[0]
        assert (plotted.linkage_rate_mean+plotted.linkage_rate_std).max()<=ax.get_xlim()[1]
        assert (plotted.balanced_accuracy_mean-plotted.balanced_accuracy_std).min()>=ax.get_ylim()[0]
        assert (plotted.balanced_accuracy_mean+plotted.balanced_accuracy_std).max()<=ax.get_ylim()[1]
    polished(ax)
    h,l=ax.get_legend_handles_labels()
    # Utility references are intentionally not assigned a privacy coordinate.
    order=[2,3,0,1] if l[0]=='I-GDP' else list(range(len(l)))
    fig.legend([h[i] for i in order],[l[i] for i in order],loc='lower center',
               ncol=1,frameon=False,bbox_to_anchor=(.57,-.012),fontsize=7)
    fig.subplots_adjust(left=.17,right=.98,top=.97,bottom=.40)
    save(fig,dataset.lower()+'_privacy_utility',out)

def lfwa_bars(raw,attacks,out):
    a=stats(attacks[attacks.method==NAA],['noise_scale','attack'],['linkage_rate'])
    levels=np.sort(raw.loc[raw.method==NAA,'noise_scale'].unique())
    assert len(levels)==10 and levels.min()>0
    fig,ax=plt.subplots(figsize=(6.9,2.5));xx=np.arange(len(levels));width=.23
    for j,attack in enumerate(['MP','AM','OP']):
        d=a[a.attack==attack].sort_values('noise_scale')
        np.testing.assert_allclose(d.noise_scale,levels,rtol=0,atol=1e-12)
        ax.bar(xx+(j-1)*width,d.linkage_rate_mean*100,width=.215,
          yerr=d.linkage_rate_std*100,color=COLORS[attack],label=attack,
          capsize=2,error_kw={'elinewidth':.7,'capthick':.7})
    ax.axhline(10,color='#8B9095',ls='--',lw=.8)
    ax.set_xticks(xx,[f'{v:.3f}' for v in levels])
    upper=max(65,5*np.ceil(100*(a.linkage_rate_mean+a.linkage_rate_std).max()/5))
    ax.set(xlabel='Anchor-noise scale, $v$',ylabel='Identity linkage (%)',ylim=(0,upper))
    ax.legend(frameon=False,ncol=3,loc='upper right')
    polished(ax,'y');fig.subplots_adjust(left=.08,right=.995,top=.97,bottom=.21)
    save(fig,'lfwa_anchor_noise_identity_leakage',out)


def render_figure3_subfigures(inputs, out, datasets=("lfwa", "celeba"), show=True):
    labels = ["Original", "C-GDP\n(projection only)",
              "C-GDP\n(private noise)", "NAA-GDP"]
    width, height = 7.10, 3.08
    left, face, gap, top = 1.06, .572, .026, .32
    row_y = [0, .63, 1.34, 2.04]
    outputs = []
    manifest = out / "figure3_subfigures_manifest.json"
    provenance = json.loads(manifest.read_text()) if manifest.exists() else {}
    style = {"font.family": "DejaVu Sans", "font.size": 8,
             "pdf.fonttype": 42, "ps.fonttype": 42,
             "savefig.facecolor": "white"}
    with plt.rc_context(style):
        for dataset in datasets:
            source = inputs / f"{dataset}_gallery.npz"
            source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
            with np.load(source, allow_pickle=False) as archive:
                images = archive["images"].copy()
                rates = archive["linkage_rates"].copy()
            assert images.shape == (4, 10, 3, 64, 64)
            assert rates.shape == (4, 10)
            assert np.isfinite(images).all()
            assert np.isfinite(rates[1:]).all()
            fig = plt.figure(figsize=(width, height), facecolor="white")
            for row in range(4):
                y_top = top + row_y[row]
                fig.text((left-.075)/width, 1-(y_top+face/2)/height,
                         labels[row], ha="right", va="center",
                         fontsize=7.5, linespacing=1.2)
                for column in range(10):
                    x = left + column*(face+gap)
                    ax = fig.add_axes([x/width, 1-(y_top+face)/height,
                                       face/width, face/height])
                    ax.imshow(np.moveaxis(images[row, column], 0, -1),
                              interpolation="nearest")
                    ax.axis("off")
                    if row:
                        fig.text((x+face/2)/width,
                                 1-(y_top+face+.022)/height,
                                 f"{100*rates[row, column]:.1f}%",
                                 ha="center", va="top", fontsize=7.4)
            for extension in ("pdf", "png"):
                target = out / f"face_reconstruction_gallery_{dataset}.{extension}"
                fig.savefig(target, dpi=230, bbox_inches="tight",
                            pad_inches=.025)
                outputs.append(target)
            plt.close(fig)
            assert hashlib.sha256(source.read_bytes()).hexdigest() == source_hash
            provenance[dataset] = {
                "input": str(source), "sha256": source_hash,
                "deployment_seed": 0, "noise_realization": 0,
                "images": 40, "displayed_linkage_rates": rates[1:].tolist(),
                "subcaption": "(a) LFWA" if dataset == "lfwa" else "(b) CelebA"}
            print(f"{dataset.upper()}: 40 saved images, 30 unchanged linkage annotations.")
            if show:
                display(Image(filename=str(out / f"face_reconstruction_gallery_{dataset}.png")))
    manifest = out / "figure3_subfigures_manifest.json"
    manifest.write_text(json.dumps(provenance, indent=2))
    outputs.append(manifest)
    return outputs


def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(8<<20),b''):
            digest.update(block)
    return digest.hexdigest()

def true_flags(values):
    lowered=values.astype(str).str.lower()
    assert lowered.isin(['true','false']).all(), 'Unrecognized convergence/final flags'
    return lowered.eq('true')

def validate_lfwa_run(folder):
    protocol=json.loads((folder/'protocol.json').read_text())
    progress=json.loads((folder/'progress.json').read_text())
    cfg=protocol['config']
    assert progress['status']=='complete' and progress['error'] is None and progress['current'] is None
    assert cfg['seeds']==list(range(5)) and cfg['realizations']==4 and cfg['participant_counts']==[5]
    np.testing.assert_allclose(cfg['anchor_scales'],np.arange(1,11)*.006,rtol=0,atol=1e-12)
    np.testing.assert_allclose(cfg['private_sigmas'],np.arange(1,11)*.3,rtol=0,atol=1e-12)
    raw=pd.read_csv(folder/'raw_results.csv')
    attacks=pd.read_csv(folder/'attack_results.csv')
    nseeds=len(cfg['seeds']);draws=cfg['realizations'];p=cfg['participant_counts'][0]
    expected_rows=nseeds*(4+draws*(len(cfg['anchor_scales'])+len(cfg['private_sigmas'])))
    expected_fits=expected_rows+nseeds*(p-1)
    assert expected_rows==protocol['expected_result_rows']==progress['scheduled_result_rows']==progress['completed_result_rows']==len(raw)
    local=raw[raw.method=='Local-only']
    assert len(local)==nseeds and (local.fit_count==0).all() and (local.aggregate_model_count==p).all()
    verified_fits=int(raw.fit_count.sum()+local.aggregate_model_count.sum())
    assert expected_fits==protocol['expected_fits']==progress['scheduled_fits']==progress['completed_fits']==verified_fits
    assert raw.run_id.is_unique and set(raw.signature)=={protocol['signature']}
    assert set(raw.outer_seed)==set(cfg['seeds']) and set(raw.participant_count)=={p}
    assert set(raw.method)=={'AA-GDP','C-GDP','I-GDP','Local-only',NAA,PRIVATE}
    assert raw.balanced_accuracy.between(0,1).all()
    assert raw[raw.method.isin(['I-GDP','Local-only'])].linkage_rate.isna().all()
    assert raw[~raw.method.isin(['I-GDP','Local-only'])].linkage_rate.between(0,1).all()
    key=['method','outer_seed','level_index','realization']
    assert not raw.duplicated(key).any()
    expected=set()
    for seed in cfg['seeds']:
        for method in ['AA-GDP','C-GDP','I-GDP','Local-only']:
            expected.add((method,seed,0,0))
        for method in [NAA,PRIVATE]:
            for level in range(1,11):
                for realization in range(draws):
                    expected.add((method,seed,level,realization))
    assert set(raw[key].itertuples(index=False,name=None))==expected
    for method,levels in [(NAA,cfg['anchor_scales']),(PRIVATE,cfg['private_sigmas'])]:
        d=raw[raw.method==method]
        np.testing.assert_allclose(d.noise_scale,[levels[int(i)-1] for i in d.level_index],rtol=0,atol=1e-12)
    assert (raw[~raw.method.isin([NAA,PRIVATE])].noise_scale==0).all()
    assert not attacks.duplicated(['run_id','attack']).any()
    assert set(attacks.run_id)<=set(raw.run_id)
    anchor=attacks[attacks.method==NAA]
    assert len(anchor)==nseeds*draws*len(cfg['anchor_scales'])*3 and set(anchor.attack)=={'MP','AM','OP'}
    for attack in ['MP','AM','OP']:
        a=anchor[anchor.attack==attack]
        joined=raw[raw.method==NAA].merge(a,on='run_id',suffixes=('_raw','_attack'),validate='one_to_one')
        assert len(joined)==nseeds*draws*len(cfg['anchor_scales'])
        for field in ['method','outer_seed','participant_count','realization']:
            assert (joined[field+'_raw']==joined[field+'_attack']).all()
        np.testing.assert_allclose(joined.noise_scale_raw,joined.noise_scale_attack,rtol=0,atol=1e-12)
        if attack=='OP':
            np.testing.assert_allclose(joined.linkage_rate_raw,joined.linkage_rate_attack,rtol=0,atol=1e-12)
    assert attacks.linkage_rate.between(0,1).all()
    saved=pd.read_csv(folder/'summary.csv')
    ls=summary(raw)
    merged=ls.merge(saved,on=['method','participant_count','noise_scale'],suffixes=('_plot','_saved'),validate='one_to_one')
    assert len(merged)==len(ls)==len(saved) and (saved.n_seeds==nseeds).all() and true_flags(saved.final).all()
    for metric in ['balanced_accuracy','linkage_rate']:
        np.testing.assert_allclose(merged[metric+'_mean_plot'],100*merged[metric+'_mean_saved'],rtol=0,atol=1e-10,equal_nan=True)
        np.testing.assert_allclose(merged[metric+'_std'],100*merged[metric+'_seed_sd'],rtol=0,atol=1e-10,equal_nan=True)
    attack_stats=stats(anchor,['noise_scale','attack'],['linkage_rate'])
    saved_attacks=pd.read_csv(folder/'attack_summary.csv')
    saved_attacks=saved_attacks[saved_attacks.method==NAA]
    joined=attack_stats.merge(saved_attacks,on=['noise_scale','attack'],suffixes=('_plot','_saved'),validate='one_to_one')
    assert len(joined)==len(attack_stats)==len(saved_attacks)==30
    assert (saved_attacks.linkage_rate_count==nseeds).all() and true_flags(saved_attacks.final).all()
    for suffix in ['mean','std']:
        np.testing.assert_allclose(joined['linkage_rate_'+suffix+'_plot'],joined['linkage_rate_'+suffix+'_saved'],rtol=0,atol=1e-12)
    for field in ['linkage_rate_mean','linkage_rate_std']:
        attack_stats[field]*=100
    diag=pd.read_csv(folder/'alignment_diagnostics.csv')
    assert diag.run_id.is_unique
    checked=raw[raw.method==NAA].merge(diag,on='run_id',suffixes=('_raw','_diag'),validate='one_to_one')
    assert len(checked)==len(diag)==nseeds*draws*len(cfg['anchor_scales'])
    np.testing.assert_array_equal(true_flags(checked.gpm_converged_raw),true_flags(checked.gpm_converged_diag))
    np.testing.assert_array_equal(checked.gpm_iterations_raw,checked.gpm_iterations_diag)
    converged=true_flags(diag.gpm_converged)
    assert (diag.loc[~converged,'gpm_iterations']==cfg['gpm_max_iters']).all()
    gpm_summary=diag.assign(converged=converged).groupby('noise_scale').agg(
        alignments=('run_id','size'),converged=('converged','sum'),
        iterations_min=('gpm_iterations','min'),iterations_max=('gpm_iterations','max'),
        last_step_mean=('gpm_last_step','mean'),alignment_seconds_mean=('alignment_seconds','mean')).reset_index()
    gpm_summary['capped']=gpm_summary.alignments-gpm_summary.converged
    evidence={'status':progress['status'],'protocol_signature':protocol['signature'],
        'protocol_code_sha256':protocol['code_sha256'],'expected_result_rows':expected_rows,
        'verified_result_rows':len(raw),'expected_fits':expected_fits,'verified_fits':verified_fits,
        'attack_rows':len(attacks),'anchor_attack_rows':len(anchor),'seeds':cfg['seeds'],
        'realizations_per_positive_level':draws,'participant_count':p,
        'anchor_scales':cfg['anchor_scales'],'private_sigmas':cfg['private_sigmas'],
        'statistics':'Average four realizations within each deployment seed, then report mean and sample SD across five seeds; rates expressed as percentages.',
        'saved_summary_agrees':True,'saved_attack_summary_agrees':True,
        'gpm_converged':int(converged.sum()),'gpm_capped':int((~converged).sum()),
        'gpm_max_iters':cfg['gpm_max_iters'],'gpm_tolerance':cfg['gpm_tolerance']}
    return raw,attacks,ls,attack_stats,gpm_summary,evidence

def validate_lfwa_gallery(inputs,out,folder,raw):
    source=inputs/'lfwa_gallery.npz'
    manifest_path=inputs/'lfwa_gallery_manifest.json'
    manifest=json.loads(manifest_path.read_text())
    deployment_path=folder/'deployments'/'seed_0'/'deployment_manifest.json'
    deployment=json.loads(deployment_path.read_text())
    assert manifest['dataset']=='lfwa' and manifest['seed']==0 and manifest['participant_count']==5
    assert manifest['target_participant']==1 and manifest['query_indices']==list(range(10))
    assert len(set(manifest['query_identities']))==10
    assert Path(manifest['score_source']).parent.name=='lfwa_full_sweep'
    for key in ['projection_sha256','data_sha256']:
        assert manifest[key]==deployment[key]
    assert manifest['deployment_hash']==deployment['config_hash']
    matched=raw[(raw.outer_seed==0)&(raw.realization==0)&(raw.participant_count==5)]
    assert set(matched.deployment_hash)=={manifest['deployment_hash']}
    with np.load(source,allow_pickle=False) as archive:
        images=archive['images'];rates=archive['linkage_rates']
        assert images.shape==(4,10,3,64,64) and rates.shape==(4,10)
        assert np.isfinite(images).all() and images.min()>=0 and images.max()<=1
        assert np.isfinite(rates[1:]).all()
        np.testing.assert_allclose(archive['anchor_scales'],np.arange(1,11)*.006,rtol=0,atol=1e-12)
        np.testing.assert_allclose(archive['private_sigmas'],np.arange(1,11)*.3,rtol=0,atol=1e-12)
        zero=matched[matched.method=='C-GDP']
        assert len(zero)==1 and zero.iloc[0].run_id==manifest['zero_run_id']
        assert zero.iloc[0].level_index==0 and zero.iloc[0].noise_scale==0
        np.testing.assert_allclose(rates[1],zero.iloc[0].linkage_rate,rtol=0,atol=1e-12)
        np.testing.assert_allclose(manifest['zero_linkage_rate'],zero.iloc[0].linkage_rate,rtol=0,atol=1e-12)
        assert len(manifest['conditions'])==10
        for column,condition in enumerate(manifest['conditions']):
            assert condition['column']==column+1 and condition['query_index']==column
            assert condition['level_index']==column+1 and condition['realization']==0
            for row,method,prefix,scale in [(2,PRIVATE,'private',condition['sigma']),(3,NAA,'anchor',condition['v'])]:
                record=matched[(matched.method==method)&(matched.level_index==column+1)]
                assert len(record)==1 and record.iloc[0].run_id==condition[prefix+'_run_id']
                np.testing.assert_allclose(record.iloc[0].noise_scale,scale,rtol=0,atol=1e-12)
                np.testing.assert_allclose(scale,archive['private_sigmas' if row==2 else 'anchor_scales'][column],rtol=0,atol=1e-12)
                np.testing.assert_allclose(rates[row,column],record.iloc[0].linkage_rate,rtol=0,atol=1e-12)
                np.testing.assert_allclose(condition[prefix+'_linkage_rate'],record.iloc[0].linkage_rate,rtol=0,atol=1e-12)
    old_manifest=out/'figure3_subfigures_manifest.json'
    previous_hash_checked=False
    if old_manifest.exists():
        previous=json.loads(old_manifest.read_text()).get('lfwa')
        if previous:
            assert previous['sha256']==sha256_file(source), 'Saved Figure 3 NPZ hash changed'
            np.testing.assert_allclose(previous['displayed_linkage_rates'],rates[1:],rtol=0,atol=1e-12)
            previous_hash_checked=True
    return {'npz_sha256':sha256_file(source),'manifest_sha256':sha256_file(manifest_path),
        'deployment_manifest_sha256':sha256_file(deployment_path),'deployment_hash':manifest['deployment_hash'],
        'data_sha256':manifest['data_sha256'],'projection_sha256':manifest['projection_sha256'],
        'seed':0,'realization':0,'verified_saved_scores':21,'displayed_annotations':30,
        'previous_figure3_hash_checked':previous_hash_checked,
        'score_source':manifest['score_source'],'annotation_scope':manifest['linkage_annotation']}

def refresh_lfwa_figures(project,download_folder=Path('/content'),show=True):
    project=Path(project)
    folder=project/'results'/'lfwa_full_sweep'
    inputs=project/'results'/'figures'/'manuscript_inputs'
    out=project/'results'/'figures'/'manuscript'
    raw,attacks,ls,attack_stats,gpm_summary,evidence=validate_lfwa_run(folder)
    gallery_evidence=validate_lfwa_gallery(inputs,out,folder,raw)
    # All result and gallery checks pass before any manuscript output is replaced.
    source_paths=[folder/name for name in ['protocol.json','progress.json','raw_results.csv',
        'attack_results.csv','summary.csv','attack_summary.csv','alignment_diagnostics.csv']]
    source_paths.extend([inputs/'lfwa_gallery.npz',inputs/'lfwa_gallery_manifest.json'])
    counts=[]
    for seed in evidence['seeds']:
        path=folder/'deployments'/f'seed_{seed}'/'deployment_manifest.json'
        deployment=json.loads(path.read_text())
        assert set(raw[raw.outer_seed==seed].deployment_hash)=={deployment['config_hash']}
        counts.extend([{'dataset':'LFWA','outer_seed':seed,**row} for row in deployment['sample_counts']])
        source_paths.append(path)
    out.mkdir(parents=True,exist_ok=True)
    outputs=[]
    for name,frame in [('lfwa_plot_statistics.csv',ls),('lfwa_attack_plot_statistics.csv',attack_stats),
                        ('lfwa_gpm_plot_statistics.csv',gpm_summary),('lfwa_sample_counts.csv',pd.DataFrame(counts))]:
        path=out/name;frame.to_csv(path,index=False);outputs.append(path)
    tradeoff(ls,5,'LFWA',out)
    lfwa_bars(raw,attacks,out)
    for name in ['lfwa_privacy_utility','lfwa_anchor_noise_identity_leakage']:
        outputs.extend([out/f'{name}.{extension}' for extension in ['pdf','png']])
    outputs.extend(render_figure3_subfigures(inputs,out,datasets=('lfwa',),show=show))
    manifest={'created_utc':datetime.now(timezone.utc).isoformat(),
        'scope':'LFWA figure refresh from saved complete results only; no fitting or CelebA rendering.',
        'result_source':str(folder),'validation':evidence,'gallery_validation':gallery_evidence,
        'sources':{str(path):sha256_file(path) for path in source_paths},
        'outputs':{path.name:sha256_file(path) for path in outputs}}
    manifest_path=out/'lfwa_full_sweep_figures_manifest.json'
    manifest_path.write_text(json.dumps(manifest,indent=2)+'\n');outputs.append(manifest_path)
    bundle=out/'lfwa_full_sweep_figures.zip'
    with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in outputs: archive.write(path,path.name)
        for path in source_paths:
            # Large saved face data stay in Drive; compact metadata and raw scores are bundled.
            if path.suffix!='.npz': archive.write(path,'provenance/'+str(path.relative_to(project)))
    download_folder=Path(download_folder)
    download_folder.mkdir(parents=True,exist_ok=True)
    destination=download_folder/bundle.name
    if bundle.resolve()!=destination.resolve(): shutil.copyfile(bundle,destination)
    print(f'Validated {evidence["verified_result_rows"]} result rows and {evidence["verified_fits"]} completed fits; regenerated three LFWA figures.')
    print(f'GPM: {evidence["gpm_converged"]} converged, {evidence["gpm_capped"]} reached {evidence["gpm_max_iters"]} iterations. Flags retained in the CSV and manifest.')
    print(f'Saved: {bundle}\nDownload from the Colab Files panel: {destination}')
    if show:
        display(Image(filename=str(out/'lfwa_privacy_utility.png')))
        display(Image(filename=str(out/'lfwa_anchor_noise_identity_leakage.png')))
    return manifest

if __name__=='__main__':
    lfwa_refresh_manifest=refresh_lfwa_figures(Path('/content/drive/MyDrive/Noisy Anchor Alignment'))
